In [ ]:
'''
python version 3.10.12
'''

In [ ]:
'''
Make sure to confirm the full path to the requirements.txt file. 
'''
! pip install -r requirements.txt
'''
Please restart this file after executing this cell !!!
'''

In [1]:
import argparse
import json
import logging
import os.path
import shutil
from collections import defaultdict

import numpy as np
import torch
from torch import nn, Tensor
from torch.utils.data import DataLoader, Subset
from torchcontrib.optim import SWA
from tqdm import tqdm

from aasist import AASIST
from aasist.data_utils import genSpoof_list, ASVspoof2019_speaker_raw,mydata_raw
from loss import SAMO, OCSoftmax
from utils import setup_seed, seed_worker, cosine_annealing, adjust_learning_rate, em, compute_eer_tdcf


In [ ]:


def init_params():
    parser = argparse.ArgumentParser(description=__doc__)

    parser.add_argument('--seed', type=int, help="random number seed", default=10)

    # Data folder prepare
    parser.add_argument("-d", "--path_to_database", type=str, help="dataset path",
                        default='/data2/sivan/')
    parser.add_argument("-p", "--path_to_protocol", type=str, help="protocol path",
                        default='../protocols/')
    parser.add_argument("-o", "--out_fold", type=str, help="output folder", required=False, default='./models/try/')
    parser.add_argument("--overwrite", action='store_true', help="overwrite output folder")

    # Dataset prepare
    parser.add_argument("--enc_dim", type=int, help="encoding dimension", default=160)

    # Training hyperparameters
    parser.add_argument('--num_epochs', type=int, default=100, help="Number of epochs for trag")
    parser.add_argument('--batch_size', type=int, default=23, help="Mini batch size for training")
    parser.add_argument('--lr', type=float, default=0.0001, help="learning rate")
    parser.add_argument('--lr_min', type=float, default=0.000005, help="min learning rate in cosine annealing")
    parser.add_argument('--lr_decay', type=float, default=0.95, help="decay learning rate for exponential schedule")
    parser.add_argument('--interval', type=int, default=1, help="interval to decay lr for exponential schedule")
    parser.add_argument("--scheduler", type=str, default="cosine2", choices=["cosine", "cosine2", "exp", "clr"])
    parser.add_argument('--beta_1', type=float, default=0.9, help="bata_1 for Adam")
    parser.add_argument('--beta_2', type=float, default=0.999, help="beta_2 for Adam")
    parser.add_argument('--eps', type=float, default=1e-8, help="epsilon for Adam")
    parser.add_argument("--gpu", type=str, help="GPU index", default="0")
    parser.add_argument('--num_workers', type=int, default=0, help="number of workers")

    # Loss setups
    parser.add_argument('-l', '--loss', type=str, default="samo",
                        choices=["softmax", "ocsoftmax", "samo"], help="loss for training")
    parser.add_argument('--num_centers', type=int, default=20,
                        help="number of centers for the sub-center one-class loss")
    parser.add_argument('--initialize_centers', type=str, default="one_hot",
                        choices=["randomly", "evenly", "one_hot", "uniform"])
    parser.add_argument('--m_real', type=float, default=0.7, help="m_real for ocsoftmax/samo loss")
    parser.add_argument('--m_fake', type=float, default=0, help="m_fake for ocsoftmax/samo loss")
    parser.add_argument('--alpha', type=float, default=20, help="scale factor for ocsoftmax loss")

    # Other
    parser.add_argument('--continue_training', action='store_true', help="continue training with trained model")
    parser.add_argument('--checkpoint', type=int, help="continue from which epoch")
    parser.add_argument('--test_on_eval', action='store_true',
                        help="whether to run EER on the evaluation set")
    parser.add_argument('--final_test', action='store_true',
                        help="whether to run best model EER on test set")
    parser.add_argument('--test_interval', type=int, default=5, help="test on eval for every how many epochs")
    parser.add_argument('--save_interval', type=int, default=5, help="save checkpoint model for every how many epochs")

    # Test setups
    parser.add_argument('--test_only', action='store_true', help='whether to run test once on chosen model')
    parser.add_argument("--test_model", type=str, default="./models/anti-spoofing_feat_model.pt")
    parser.add_argument("--scoring", type=str, default=None, choices=["fc", "samo", "ocsoftmax"])
    parser.add_argument("--save_score", type=str, default=None,
                        help='score file name to save individual score for each sample')
    parser.add_argument('--save_center', action='store_true', help='whether to save centers as log files')
    parser.add_argument('--dp', action='store_true', default=False, help='use Data Parallel')
    parser.add_argument('--one_hot', action='store_true', help='use one hot vectors in final test')

    # Scenario setups for SAMO
    parser.add_argument('--train_sp', type=int, default=1,
                        help="1: speaker-aware loss(sim score); "
                             "2: speaker-agnostic(maxscore); "
                             "0: other loss")
    # 1: 1 on 1 similarity for all training data
    # 2: maxscore for enrolled training centers
    parser.add_argument('--val_sp', type=int, default=1,
                        help="1: speaker-aware loss(sim score); "
                             "2: speaker-agnostic(maxscore); "
                             "0: speaker-independent(use current train centers)")
    # 0, 1, 2 all based on maxscore with all centers
    # 1 replace score with specific distance for target data, keep maxscore for non-target
    # 0 and 2 differ only for enrollment, 0 enrolls training centers, 2 enrolls test centers
    parser.add_argument('--target', type=int, default=1, help='load target speaker data only in val and eval')
    parser.add_argument('--update_interval', type=int, default=3, help="update training centers for every x epochs")
    parser.add_argument('--init_center', type=int, default=1, help='initialized center with orthogonal embeds')
    parser.add_argument('--update_stop', type=int, default=None, help='stop updating centers after x epochs')
    parser.add_argument("--center_sampler", type=str, default="sequential", choices=["sequential", "random"])

    args = parser.parse_args()

    if not args.dp:
        os.environ["CUDA_VISIBLE_DEVICES"] = args.gpu
    if args.update_stop == None:
        args.update_stop = args.num_epochs + 1

    # Set seeds
    setup_seed(args.seed)

    if args.continue_training:
        pass
    else:
        # Path for output data
        if os.path.exists(args.out_fold):
            logging.warning('{} exists'.format(args.out_fold))
            print("overwrite:{}".format(args.overwrite))
        if args.out_fold == './models/try/':
            args.overwrite = True
        os.makedirs(args.out_fold, exist_ok=args.overwrite)
        if args.overwrite:
            shutil.rmtree(args.out_fold)
            os.mkdir(args.out_fold)

        # Folder for intermediate results
        if not os.path.exists(os.path.join(args.out_fold, 'checkpoint')):
            os.makedirs(os.path.join(args.out_fold, 'checkpoint'))
        else:
            shutil.rmtree(os.path.join(args.out_fold, 'checkpoint'))
            os.mkdir(os.path.join(args.out_fold, 'checkpoint'))

        # Path for input data
        if not os.path.exists(args.path_to_database):
            raise RuntimeError(f'Path {args.path_to_database} does not exists!')

        # Save training arguments
        with open(os.path.join(args.out_fold, 'args.json'), 'w') as file:
            file.write(json.dumps(vars(args), sort_keys=True, separators=('\n', ':')))
        with open(os.path.join(args.out_fold, 'train_loss.log'), 'w') as file:
            file.write("Start recording training loss ...\n")
        with open(os.path.join(args.out_fold, 'dev_loss.log'), 'w') as file:
            file.write("Start recording validation loss ...\n")
        with open(os.path.join(args.out_fold, 'test_loss.log'), 'w') as file:
            file.write("Start recording test loss ...\n")

    args.cuda = torch.cuda.is_available()
    print('Cuda device available: ', args.cuda)
    args.device = torch.device("cuda" if args.cuda else "cpu")

    return args

def get_loader(eval_path,path_to_database, seed, target, batch_size,eval_trial_path):
   
# def get_loader(args):
    """
    Make PyTorch DataLoaders for train / developement / evaluation
    Adapted from https://github.com/clovaai/aasist
    """
    center_sampler = "sequential"
    database_path = path_to_database
    seed = seed
    target_only = target
    batch_size = batch_size

    trn_database_path = os.path.join(database_path + "LA/ASVspoof2019_LA_train/")
    dev_database_path = os.path.join(database_path + "LA/ASVspoof2019_LA_dev/")
    eval_database_path = os.path.join(database_path + "LA/ASVspoof2019_LA_eval/")
   

    trn_list_path = os.path.join(path_to_database,"LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.train.trn.txt")
    dev_trial_path = os.path.join(path_to_database,"LA/ASVspoof2019_LA_cm_protocols/ASVspoof2019.LA.cm.dev.trl.txt")
    
    # eval_trial_path = "protocols/ASVspoof2019.LA.cm.eval.trl.txt"
    
    dev_enroll_path = [os.path.join(path_to_database,"LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.female.trn.txt"),
                       os.path.join(path_to_database,"LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.dev.male.trn.txt")]
    eval_enroll_path = [os.path.join(path_to_database,"LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.female.trn.txt"),
                        os.path.join(path_to_database,"LA/ASVspoof2019_LA_asv_protocols/ASVspoof2019.LA.asv.eval.male.trn.txt")]

    # Read all training data
    label_trn, file_train, utt2spk_train, tag_train = genSpoof_list(dir_meta=trn_list_path, enroll=False, train=True)
    trn_centers = len(set(utt2spk_train.values()))
    print("no. training files:", len(file_train))
    print("no. training speakers:", trn_centers)
    train_set = ASVspoof2019_speaker_raw(list_IDs=file_train,
                                         labels=label_trn,
                                         utt2spk=utt2spk_train,
                                         base_dir=trn_database_path,
                                         tag_list=tag_train,
                                         train=True)
    gen = torch.Generator()
    gen.manual_seed(seed)
    trn_loader = DataLoader(train_set,
                            batch_size=batch_size,
                            shuffle=True,
                            drop_last=True,
                            pin_memory=True,
                            worker_init_fn=seed_worker,
                            generator=gen)

    # Read bona-fide-only training data
    num_bonafide_train = 2580
    train_set_fix = ASVspoof2019_speaker_raw(list_IDs=file_train,
                                             labels=label_trn,
                                             utt2spk=utt2spk_train,
                                             base_dir=trn_database_path,
                                             tag_list=tag_train,
                                             train=False)
    trn_bona_set = Subset(train_set_fix, range(num_bonafide_train))
    if center_sampler == "random":
        trn_bona = DataLoader(trn_bona_set,
                              batch_size=batch_size,
                              shuffle=True,
                              drop_last=False,
                              pin_memory=True,
                              worker_init_fn=seed_worker,
                              generator=gen)
        '''default'''
    elif center_sampler == "sequential":
        trn_bona = DataLoader(trn_bona_set,
                              batch_size=int(batch_size),
                              shuffle=False,
                              drop_last=False,
                              pin_memory=True
                              # sampler=torch_sampler.SequentialSampler(range(num_bonafide_train))
                              )
    else:
        raise NotImplementedError

    # Read dev enrollment data
    label_dev_enroll, file_dev_enroll, utt2spk_dev_enroll, tag_dev_enroll = genSpoof_list(dir_meta=dev_enroll_path,
                                                                                          enroll=True,
                                                                                          train=False)
    dev_enroll_spk = set(utt2spk_dev_enroll.values())
    dev_centers = len(dev_enroll_spk)
    print(f"no. validation enrollment files: {len(file_dev_enroll)}")
    print(f"no. validation enrollment speakers: {dev_centers}")
    dev_set_enroll = ASVspoof2019_speaker_raw(list_IDs=file_dev_enroll,
                                              labels=label_dev_enroll,
                                              utt2spk=utt2spk_dev_enroll,
                                              base_dir=dev_database_path,
                                              tag_list=tag_dev_enroll,
                                              train=False)
    dev_enroll = DataLoader(dev_set_enroll,
                            batch_size=batch_size,
                            shuffle=False,
                            drop_last=False,
                            pin_memory=True)

    # Read target-only dev data
    label_dev, file_dev, utt2spk_dev, tag_dev = genSpoof_list(dir_meta=dev_trial_path, enroll=False, train=False,
                                                              target_only=target_only, enroll_spk=dev_enroll_spk)
    print(f"no. validation files: {len(file_dev)}")
    dev_set = ASVspoof2019_speaker_raw(list_IDs=file_dev,
                                       labels=label_dev,
                                       utt2spk=utt2spk_dev,
                                       base_dir=dev_database_path,
                                       tag_list=tag_dev,
                                       train=False)
    dev_loader = DataLoader(dev_set,
                            batch_size=batch_size,
                            shuffle=False,
                            drop_last=False,
                            pin_memory=True)

    # Read eval enrollment data
    label_eval_enroll, file_eval_enroll, utt2spk_eval_enroll, tag_eval_enroll = genSpoof_list(dir_meta=eval_enroll_path,
                                                                                              enroll=True,
                                                                                              train=False)
    eval_enroll_spk = set(utt2spk_eval_enroll.values())
    eval_centers = len(eval_enroll_spk)
    print(f"no. eval enrollment files: {len(file_eval_enroll)}")
    print(f"no. eval enrollment speakers: {eval_centers}")
    eval_set_enroll = ASVspoof2019_speaker_raw(list_IDs=file_eval_enroll,
                                               labels=label_eval_enroll,
                                               utt2spk=utt2spk_eval_enroll,
                                               base_dir=eval_database_path,
                                               tag_list=tag_eval_enroll,
                                               train=False)
    eval_enroll = DataLoader(eval_set_enroll,
                             batch_size=batch_size,
                             shuffle=False,
                             drop_last=False,
                             pin_memory=True)

   
    label_eval, file_eval, utt2spk_eval, tag_eval = genSpoof_list(dir_meta=eval_trial_path, enroll=False, train=False,
                                                                  target_only=target_only, enroll_spk=eval_enroll_spk)
    # print(f"no. eval files: {len(file_eval)}")
  
    eval_set = mydata_raw(list_IDs=file_eval,
                                        labels=label_eval,
                                        utt2spk=utt2spk_eval,
                                        # base_dir=eval_database_path,
                                        base_dir=eval_path,
                                        tag_list=tag_eval,
                                        train=False)
    eval_loader = DataLoader(eval_set,
                             batch_size=batch_size,
                             shuffle=False,
                             drop_last=False,
                             pin_memory=True)

    num_centers = [trn_centers, dev_centers, eval_centers]

    return trn_loader, dev_loader, eval_loader, trn_bona, dev_enroll, eval_enroll, num_centers


def get_model(model_config,device):
    _model = AASIST.Model
    feat_model = _model(model_config).to(device)
    nb_params = sum([param.view(-1).size()[0] for param in feat_model.parameters()])
    print("no. model params:{}".format(nb_params))
    return feat_model




def update_embeds(device, enroll_model, loader):
    enroll_emb_dict = {}
    with torch.no_grad():
        for i, (batch_x, _, spk, _, _) in enumerate(tqdm(loader)):  # batch_x = x_input, key = utt_list
            batch_x = batch_x.to(device)
            batch_cm_emb, _ = enroll_model(batch_x)
            batch_cm_emb = batch_cm_emb.detach().cpu().numpy()

            for s, cm_emb in zip(spk, batch_cm_emb):
                if s not in enroll_emb_dict:
                    enroll_emb_dict[s] = []

                enroll_emb_dict[s].append(cm_emb)

        for spk in enroll_emb_dict:
            enroll_emb_dict[spk] = Tensor(np.mean(enroll_emb_dict[spk], axis=0))

    return enroll_emb_dict


def test(path_to_database,test_model,eval_trial_path,voicewukongpath,save_path):

    seed=10
    batch_size=24
    scoring='samo'
    val_sp=0
    one_hot=0
    cuda=1
    
    torch.set_default_tensor_type(torch.FloatTensor)
    
    device = torch.device("cuda" if cuda else "cpu")
    target=0
    
        # load models
    if test_model[-3:] == "pth":
        with open("aasist/AASIST.conf", "r") as f_json:
            config = json.loads(f_json.read())
        feat_model = get_model(config["model_config"],device)
        feat_model.load_state_dict(
            torch.load(test_model, map_location=device))
    else:
        feat_model = torch.load(test_model).to(device)
    print("Model loaded : {}".format(test_model))

    print("Start evaluation...")

    feat_model.eval()

    # load test data and initialize loss
    # reverse [0,1] label when loading aasist pretrain
    _, _, eval_data_loader, train_bona_loader, _, eval_enroll_loader, _ = get_loader(eval_path=voicewukongpath,
                                                                                        path_to_database=path_to_database,
                                                                                     seed=seed, 
                                                                                     target=target, 
                                                                                     batch_size=batch_size,
                                                                                     eval_trial_path=eval_trial_path)
    '''eval_enroll_loader not use'''
    enc_dim=160
    m_real=0.7
    m_fake=0
    alpha=20
    
    samo = SAMO(enc_dim, m_real=m_real, m_fake=m_fake, alpha=alpha).to(device)

    with torch.no_grad():

        ip1_loader, utt_loader, idx_loader, score_loader, spk_loader, tag_loader = [], [], [], [], [], []

        if scoring == "samo":
            if val_sp:
                # define and update eval centers
                eval_enroll = update_embeds(device, feat_model, eval_enroll_loader)
                
            else:  # use training centers without eval enrollment
                
                eval_enroll = update_embeds(device, feat_model, train_bona_loader)
            samo.center = torch.stack(list(eval_enroll.values()))
     
        for i, (feat, labels, spk, utt, tag) in enumerate(tqdm(eval_data_loader)):
            feat = feat.to(device)
            labels = labels.to(device)
            feats, feat_outputs = feat_model(feat)

            # loss cal
            if scoring == "samo":
                if target:  # loss calculation for target-only speakers
                    # val_sp = 0 or 2 calculate all maxscore only
                    # val_sp = 1 calculate 1 on 1 scores
                    _, score = samo(feats, labels, spk, eval_enroll, val_sp)
                     
                else:
                    _, score = samo.inference(feats, labels, spk, eval_enroll, val_sp)
            elif scoring == "fc":
                if test_model[-3:] == "pth":
                    score = feat_outputs[:, 1]  # pretrained networks with reversed labels
                else:
                    score = feat_outputs[:, 0]  # samo pretrained
           

            ip1_loader.append(feats)
            idx_loader.append(labels)
            score_loader.append(score)
            utt_loader.extend(utt)
            spk_loader.extend(spk)
            tag_loader.extend(tag)
 
        scores = torch.cat(score_loader, 0).data.cpu().numpy()
        labels = torch.cat(idx_loader, 0).data.cpu().numpy()
        # eer = em.compute_eer(scores[labels == 0], scores[labels == 1])[0]

    # if args.save_score != None:
    with open(save_path, "w") as fh:  # w as in overwrite mode
        for utt, tag, score, label, spk in zip(utt_loader, tag_loader, scores, labels, spk_loader):
            fh.write(f"{utt} {tag} {label} {score} {spk}\n")
    print(f"Scores saved to {save_path}")

  


if __name__ == "__main__":
    # args = init_params()
    '''
    |- path to VoiceWukong dataset
        |- Alldataset
        |- Alldataset32K
        |- ...
    '''
    eval_path='change this to your VoiceWukong path'
    '''
    ASVSpoof2019 dataset path/
       |- LA
          |- ASVspoof2021_LA_eval/flac
          |- ASVspoof2019_LA_train/flac
          |- ASVspoof2019_LA_dev/flac
    '''
    path_to_database='change this to your ASVSpoof2019 dataset path ' # download here [https://datashare.ed.ac.uk/handle/10283/3336]
    test_model='change to the path to Samo model' # download here [https://huggingface.co/VoiceWukong/VoiceWukong/resolve/main/samo.pt?download=true]
    
    zh_save_path = "change to the path to save your SAMO zh_eval_score.txt"
    en_save_path= "change to the path to save your SAMO en_eval_score.txt"
    zh_eval_list='change to the path to zh_eval_list.txt'
    en_eval_list='change to the path to eval_list.txt'
    test(path_to_database=path_to_database,test_model=test_model,eval_trial_path=zh_eval_list,voicewukongpath=eval_path,save_path=zh_save_path)
    test(path_to_database=path_to_database,test_model=test_model,eval_trial_path=en_eval_list,voicewukongpath=eval_path,save_path=en_save_path)
    
